# 05 - Baseline k-means (DS, Day 8)

The DA plan's Day 8 task for DS: **run a baseline k-means clustering**. A baseline is the simple first attempt that later models (Day 9) must beat.

It answers four questions:
1. How many clusters does the data support? (inertia and silhouette for k = 2 to 12)
2. What do the clusters look like? (sizes, and which features define each one)
3. How stable are they? (re-run with different random starts and compare)
4. Who is in each cluster, in plain units? (budget, intent, region, device ...)

**Rules**
- Input is `customer_features_kmeans.parquet` from notebook 04. Nothing else goes into the model.
- **The answer key (`_restricted/`) is never opened here.** It scores the final clustering, and only after the modelling is finished.
- Outcomes (conversion, revenue, payments) are not used. Testing segments against them is the Day 10 task.

## 0. Setup

In [ ]:
%pip install -q scikit-learn

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

ROOT = Path.cwd().parent
FEAT = ROOT / "data" / "features"
OUT = ROOT / "notebooks" / "outputs"
OUT.mkdir(exist_ok=True)
assert (FEAT / "customer_features_kmeans.parquet").exists(), "Run 04_features.ipynb first"

X = pd.read_parquet(FEAT / "customer_features_kmeans.parquet").set_index("customer_id")
assert not X.isna().any().any(), "Missing values in the model matrix: re-run 04_features"
assert all(pd.api.types.is_numeric_dtype(t) for t in X.dtypes), "Non-numeric column in the model matrix"
print(X.shape, "customers x features")

## 1. How many clusters? (inertia and silhouette)
- **Inertia** is how tight the clusters are. It always falls as k grows, so look for the "elbow" where the drop slows.
- **Silhouette** is how well separated they are (higher is better, 1 is perfect). It is computed on a random sample of 15,000 customers to keep it fast.
- Mixed data with overlapping groups gives low silhouettes. The README says to expect low values on this dataset, so use the *shape* of the curve, not the absolute number.

In [ ]:
SIL_SAMPLE = 15_000
rows = []
for k in range(2, 13):
    km = KMeans(n_clusters=k, n_init=5, random_state=0).fit(X)
    sil = silhouette_score(X, km.labels_, sample_size=min(SIL_SAMPLE, len(X)), random_state=0)
    rows.append({"k": k, "inertia": km.inertia_, "silhouette": sil})
    print(f"k={k:2d}  inertia={km.inertia_:,.0f}  silhouette={sil:.3f}")
curve = pd.DataFrame(rows).set_index("k")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(curve.index, curve["inertia"], marker="o", color="#1f4e79"); ax[0].set(title="Inertia (look for the elbow)", xlabel="k", ylabel="inertia")
ax[1].plot(curve.index, curve["silhouette"], marker="o", color="#1f4e79"); ax[1].set(title="Silhouette (higher = better separated)", xlabel="k", ylabel="silhouette")
plt.tight_layout(); plt.savefig(OUT / "kmeans_k_selection.png", dpi=150); plt.show()
curve.round(3)

## 2. CONFIG: fit the baseline
`K` is your choice, made from the curves above. The README says eight personas generated the customers, so **k = 8 is a sensible candidate to include**, but let the curves speak first. If they favour a different k, write down why you picked it.

In [ ]:
K = 8     # CONFIG: set from section 1

km = KMeans(n_clusters=K, n_init=20, random_state=0).fit(X)
labels = pd.Series(km.labels_, index=X.index, name=f"kmeans_k{K}")
sizes = labels.value_counts().sort_index().to_frame("customers")
sizes["share_%"] = (sizes["customers"] / len(labels) * 100).round(1)
print(f"k={K}: inertia {km.inertia_:,.0f}")
sizes

## 3. Stability: do the clusters survive a different random start?
k-means starts from random points, so two runs can give different clusters. We fit it 8 more times with different seeds and compare each result with the main one using the **adjusted Rand index (ARI)**: 1 means identical groupings, 0 means no better than chance.
Low ARI between runs means the clusters are not a stable feature of the data. This is **not** the answer-key score. It only compares our own runs with each other.

In [ ]:
seeds = [1, 2, 3, 4, 5, 6, 7, 8]
runs = {s: KMeans(n_clusters=K, n_init=1, random_state=s).fit_predict(X) for s in seeds}
vs_main = pd.Series({s: adjusted_rand_score(labels, l) for s, l in runs.items()}, name="ARI vs main run")
pairs = [adjusted_rand_score(runs[a], runs[b]) for i, a in enumerate(seeds) for b in seeds[i + 1:]]
print(f"ARI of each seed against the main run:\n{vs_main.round(3).to_string()}")
print(f"\nMean ARI between all pairs of runs: {np.mean(pairs):.3f}   (min {np.min(pairs):.3f}, max {np.max(pairs):.3f})")

## 4. What defines each cluster?
For every feature, each cluster's average is compared with the overall average and divided by the overall standard deviation. So **+1 means "one standard deviation above the average customer"** and 0 means average, on the same scale for every feature, including the 0/1 flags and one-hot categories.
The heatmap shows the 25 features that differ most between clusters. The printout lists, for each cluster, the features furthest from average.

In [ ]:
means = X.groupby(labels).mean()
dev = (means - X.mean()) / X.std()                      # SD from the overall average, same scale for every feature
top_feats = dev.var().sort_values(ascending=False).head(25).index.tolist()

fig, ax = plt.subplots(figsize=(11, 7))
sns.heatmap(dev[top_feats].T, cmap="RdBu_r", center=0, vmin=-3, vmax=3, annot=True, fmt=".1f", annot_kws={"size": 7}, cbar_kws={"label": "SD from average"}, ax=ax)
ax.set(title=f"Cluster profile: 25 most discriminating features (k={K}), colour capped at +/-3 SD", xlabel="cluster", ylabel="")
plt.tight_layout(); plt.savefig(OUT / "kmeans_cluster_heatmap.png", dpi=150); plt.show()

for c in dev.index:
    top = dev.loc[c].reindex(dev.loc[c].abs().sort_values(ascending=False).index).head(6)
    print(f"Cluster {c} ({sizes.loc[c, 'share_%']}% of customers): " + ", ".join(f"{f} ({v:+.1f})" for f, v in top.items()))

## 5. Who is in each cluster? (plain units)
The same clusters, described with the readable features from `customer_features.parquet`: intent, budget, overseas share, devices, what they look at.
Outcomes are deliberately not shown here. That comparison is the Day 10 task.

In [ ]:
F = pd.read_parquet(FEAT / "customer_features.parquet").set_index("customer_id").loc[X.index]
prof = pd.DataFrame(index=sorted(labels.unique()))
prof["customers"] = sizes["customers"]
prof["buy_%"] = (F["primary_intent"].eq("Buy").groupby(labels).mean() * 100).round(0)
prof["overseas_%"] = (F["is_overseas"].astype(float).groupby(labels).mean() * 100).round(0)
prof["median_budget_gbp"] = F["budget_gbp"].groupby(labels).median().round(0)
prof["budget_missing_%"] = (F["budget_missing"].groupby(labels).mean() * 100).round(0)
for c, name in [("share_mobile", "mobile_%"), ("share_night", "night_%"), ("share_new_build", "new_build_%"), ("share_offplan", "offplan_%")]:
    if c in F.columns:
        prof[name] = (F[c].groupby(labels).mean() * 100).round(0)
prof["median_sessions"] = F["n_sessions"].groupby(labels).median()
prof["top_lead_source"] = F["primary_lead_source"].groupby(labels).agg(lambda s: s.value_counts().index[0])
prof["top_country"] = F["country_iso"].groupby(labels).agg(lambda s: s.value_counts().index[0])
prof["top_region_of_residence"] = F["uk_region_of_residence"].groupby(labels).agg(lambda s: s.value_counts().index[0] if s.notna().any() else "n/a")
prof

## 6. Save the baseline labels

In [ ]:
labels.reset_index().to_parquet(FEAT / "labels_kmeans_baseline.parquet", index=False)
prof.to_csv(OUT / "kmeans_baseline_profile.csv")
curve.to_csv(OUT / "kmeans_k_selection.csv")
print("Saved labels to", FEAT / "labels_kmeans_baseline.parquet")

## 7. Validate

In [ ]:
results = {
    "every customer has exactly one cluster": labels.notna().all() and len(labels) == len(X) and labels.index.is_unique,
    "no empty cluster": (sizes["customers"] > 0).all() and len(sizes) == K,
    "no tiny cluster (under 1% of customers)": bool((sizes["share_%"] >= 1).all()),
    "runs agree reasonably (mean ARI >= 0.7)": bool(np.mean(pairs) >= 0.7),
    "answer key never read": True,
}
for k_, v in results.items():
    print(("PASS  " if v else "CHECK ") + k_)
print("\n'CHECK' is not an error. It is something to look at and write down.")

## Hand-off notes
- **Day 8 baseline:** `labels_kmeans_baseline.parquet`. The DA team does not need it today (per the plan).
- **Write down** the k you chose and why, the mean stability ARI, and any cluster you cannot explain.
- **Day 9:** try at least one model that handles mixed data natively (k-prototypes, or Gower distance with a different clusterer) on `customer_features_mixed.parquet`, and shortlist 2 to 3 candidates to share with the DA team.
- **Do not** look at the answer key or run `recoverability_check.py` until the models are final.